# Validation & Scenarios — Site Suitability Recommender

NDCG ranking validation and scenario comparison for agriculture, solar,
and conservation use cases.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.metrics import ndcg_score

## Validation & Scenario Steps

1. Load ranked sites and ground-truth expert rankings
2. Compute NDCG@k for ranking quality assessment
3. Define alternative weight profiles for each scenario
4. Re-rank sites under agriculture, solar, and conservation scenarios
5. Compare top-10 recommendations across scenarios

In [ ]:
# Load ranked sites
sites = gpd.read_file('../data/processed/sites_ranked.shp')

# NDCG validation against expert rankings
expert_scores = pd.read_csv('../data/raw/expert_rankings.csv')
merged = sites.merge(expert_scores, on='site_id')

y_true = merged['expert_score'].values.reshape(1, -1)
y_pred = merged['suitability_score'].values.reshape(1, -1)

for k in [5, 10, 20]:
    score = ndcg_score(y_true, y_pred, k=k)
    print(f'NDCG@{k}: {score:.4f}')

In [ ]:
# Scenario comparison
criteria = ['soil_quality', 'elevation', 'slope', 'ndvi', 'water_proximity', 'road_proximity']

scenario_weights = {
    'Agriculture': [0.30, 0.05, 0.10, 0.25, 0.20, 0.10],
    'Solar Farm':  [0.05, 0.15, 0.25, 0.05, 0.10, 0.40],
    'Conservation':[0.10, 0.10, 0.05, 0.40, 0.30, 0.05],
}

for scenario, weights in scenario_weights.items():
    sites[f'score_{scenario}'] = sum(
        w * sites[c] for w, c in zip(weights, criteria)
    )
    top5 = sites.nlargest(5, f'score_{scenario}')[['site_id', f'score_{scenario}']]
    print(f'\n--- {scenario} Top 5 ---')
    print(top5.to_string(index=False))

# Save scenario results
sites.to_file('../data/processed/sites_scenarios.shp')
print('\nScenario analysis complete.')